In [1]:
# Build the class dataset: the full Parquet (benign + attack) converted to one CSV.
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("build_class_dataset")
         .config("spark.driver.memory", "8g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

base = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/correlated_dataset/"
        "2026-09-26_00-28-45/finalized_data")
out  = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"

# Column order taken from the sample CSV header, so the file matches what the notebook expects
cols = spark.read.option("header", True).csv(base + "/Sample_Dataset_CSV/Benign/part-*.csv").columns

full = spark.read.parquet(base + "/Full_Dataset_Parquet/2026-*")
full = full.withColumn("datetime", F.date_format("datetime", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
full = full.select([(F.col(c).cast("string") if c in full.columns
                     else F.lit(None).cast("string")).alias(c) for c in cols])

print("Total rows:", full.count())
full.groupBy("label_tactic").count().orderBy("count", ascending=False).show(truncate=False)

full.coalesce(1).write.mode("overwrite").option("header", True).csv(out)
print("Written to", out)

26/10/26 16:33:10 WARN Utils: Your hostname, car-ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.131 instead (on interface ens33)
26/10/26 16:33:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 16:33:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/26 16:33:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Benign rows (sample): 17999
Attack rows (full):   359655
+----------------+------+
|label_tactic    |count |
+----------------+------+
|Discovery       |347650|
|Reconnaissance  |11938 |
|Lateral Movement|55    |
|Initial Access  |10    |
|Defense Evasion |2     |
+----------------+------+



[Stage 11:>                                                         (0 + 1) / 1]

Written to hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_csv
